In [1]:
import pandas as pd
import numpy as np
import requests
import streamlit as st
import sqlite3

In [68]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from io import StringIO

def scrape_cbb_data(season=2026, table_type="school_advanced"):
    """
    Valid table_types:
    - 'school_basic'
    - 'school_advanced'
    - 'opponent_basic'
    - 'opponent_advanced'
    """
    
    # 1. Map the table type to the exact URL configuration and HTML element ID
    config = {
        "school_basic": {
            "url_suffix": "school-stats.html",
            "table_id": "basic_school_stats"
        },
        "school_advanced": {
            "url_suffix": "advanced-school-stats.html",
            "table_id": "adv_school_stats"
        },
        "opponent_basic": {
            "url_suffix": "opponent-stats.html",
            "table_id": "basic_opp_stats"
        },
        "opponent_advanced": {
            "url_suffix": "advanced-opponent-stats.html",
            "table_id": "adv_opp_stats"
        }
    }
    
    if table_type not in config:
        raise ValueError(f"Invalid table_type. Choose from: {list(config.keys())}")
        
    # 2. Build the URL dynamically
    suffix = config[table_type]["url_suffix"]
    target_table_id = config[table_type]["table_id"]
    url = f"https://www.sports-reference.com/cbb/seasons/men/{season}-{suffix}"
    
    # 3. Fetch the web page
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    response = requests.get(url, headers=headers)
    
    if response.status_code != 200:
        print(f"Failed to fetch data. Status code: {response.status_code}")
        return None
        
    # 4. Parse HTML and find the specific table ID
    soup = BeautifulSoup(response.content, 'html.parser')
    table = soup.find('table', {'id': target_table_id})
    
    if table is None:
        print(f"Error: Could not find table with ID '{target_table_id}' on the page.")
        return None
        
    # 5. Read into Pandas
    df = pd.read_html(StringIO(str(table)))[0]
    
    # 6. Clean Multi-index headers
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = ['_'.join(col).strip() for col in df.columns.values]
        
    # 7. Remove repeated header rows throughout the table
    school_col = [c for c in df.columns if 'School' in c][0]
    df = df[df[school_col] != 'School']
    df = df.reset_index(drop=True)
    
    return df

# --- HOW TO RUN IT FOR ALL 4 TABLES ---

tables_to_get = ["school_basic", "school_advanced", "opponent_basic", "opponent_advanced"]

for t_type in tables_to_get:
    print(f"Fetching {t_type} data...")
    
    # Scrape the data
    data = scrape_cbb_data(season=2026, table_type=t_type)
    
    if data is not None:
        # Generate a distinct filename for each table
        filename = f"cbb_{t_type}_2026.csv"
        data.to_csv(filename, index=False)
        print(f"--> Saved {data.shape[0]} teams to {filename}")
        
    # CRITICAL: Sleep 4 seconds between tables to respect the 20 requests per minute rule!
    print("Sleeping 4 seconds to respect rate limits...")
    time.sleep(4)

print("\nAll downloads complete! Check your folder for the 4 CSV files.")

Fetching school_basic data...
--> Saved 383 teams to cbb_school_basic_2026.csv
Sleeping 4 seconds to respect rate limits...
Fetching school_advanced data...
--> Saved 383 teams to cbb_school_advanced_2026.csv
Sleeping 4 seconds to respect rate limits...


KeyboardInterrupt: 

In [2]:
df_sa = pd.read_csv("data/2026/cbb_school_advanced_2026.csv")
df_sb = pd.read_csv("data/2026/cbb_school_basic_2026.csv")
df_oa = pd.read_csv("data/2026/cbb_opponent_advanced_2026.csv")
df_ob = pd.read_csv("data/2026/cbb_opponent_basic_2026.csv")
df_ob.head()

,Unnamed: 0_level_0_Rk,Unnamed: 1_level_0_School,Overall_G,Overall_W,Overall_L,Overall_W-L%,Overall_SRS,Overall_SOS,Unnamed: 8_level_0_Unnamed: 8_level_1,Conf._W,...,Opponent_FT,Opponent_FTA,Opponent_FT%,Opponent_ORB,Opponent_TRB,Opponent_AST,Opponent_STL,Opponent_BLK,Opponent_TOV,Opponent_PF
0,1.0,Abilene Christian,33,14,19,.424,-7.38,-1.45,NaN,5,...,607,842,.721,266,1029,393,278,147,541,605
1,2.0,Air Force,32,3,29,.094,-13.75,4.60,NaN,0,...,488,671,.727,321,1146,488,255,113,325,569
2,3.0,Akron NCAA,35,29,6,.829,8.48,-2.91,NaN,17,...,444,616,.721,383,1157,467,221,100,470,547
3,4.0,Alabama NCAA,35,25,10,.714,23.03,14.57,NaN,13,...,535,769,.696,483,1395,488,224,133,334,675
4,5.0,Alabama A&M,33,18,15,.545,-13.31,-10.81,NaN,10,...,518,714,.725,329,1129,463,202,99,372,634


In [3]:
data = ["df_sa", "df_sb"]
cols = ["Opponent_FGA", "Opponent_TOV", "Opponent_ORB", "Opponent_FTA", "Points_Opp."]
totals = ["Totals_FGA", "Totals_TOV", "Totals_ORB", "Totals_FTA", "Points_Tm."]


for i in cols:
    df_ob[i] = pd.to_numeric(df_ob[i], errors='coerce')
for i in totals:
    df_sb[i] = pd.to_numeric(df_sb[i], errors='coerce')

In [4]:
df_ob["Possessions"] = df_ob["Opponent_FGA"] + df_ob["Opponent_TOV"] - df_ob["Opponent_ORB"] + (.475 * df_ob["Opponent_FTA"])
#df_ob.head()
df_ob["Def_Eff"] = df_ob["Points_Opp."] / df_ob["Possessions"]
df_ob.head()

,Unnamed: 0_level_0_Rk,Unnamed: 1_level_0_School,Overall_G,Overall_W,Overall_L,Overall_W-L%,Overall_SRS,Overall_SOS,Unnamed: 8_level_0_Unnamed: 8_level_1,Conf._W,...,Opponent_FT%,Opponent_ORB,Opponent_TRB,Opponent_AST,Opponent_STL,Opponent_BLK,Opponent_TOV,Opponent_PF,Possessions,Def_Eff
0,1.0,Abilene Christian,33,14,19,.424,-7.38,-1.45,NaN,5,...,.721,266.0,1029,393,278,147,541.0,605,2295.950,1.040528
1,2.0,Air Force,32,3,29,.094,-13.75,4.60,NaN,0,...,.727,321.0,1146,488,255,113,325.0,569,2156.725,1.185130
2,3.0,Akron NCAA,35,29,6,.829,8.48,-2.91,NaN,17,...,.721,383.0,1157,467,221,100,470.0,547,2506.600,1.033272
3,4.0,Alabama NCAA,35,25,10,.714,23.03,14.57,NaN,13,...,.696,483.0,1395,488,224,133,334.0,675,2617.275,1.106494
4,5.0,Alabama A&M,33,18,15,.545,-13.31,-10.81,NaN,10,...,.725,329.0,1129,463,202,99,372.0,634,2242.150,1.067279


In [5]:
df_sb["Possessions"] = df_sb["Totals_FGA"] + df_sb["Totals_TOV"] - df_sb["Totals_ORB"] + (.475 * df_sb["Totals_FTA"])
df_sb["Off_Eff"] = df_sb["Points_Tm."] / df_sb["Possessions"]
df_sb["Overall_G"] = pd.to_numeric(df_sb["Overall_G"], errors='coerce')
df_sb["Poss_Per_Game"] = df_sb["Possessions"] / df_sb["Overall_G"]
df_sb.head()

,Unnamed: 0_level_0_Rk,Unnamed: 1_level_0_School,Overall_G,Overall_W,Overall_L,Overall_W-L%,Overall_SRS,Overall_SOS,Unnamed: 8_level_0_Unnamed: 8_level_1,Conf._W,...,Totals_ORB,Totals_TRB,Totals_AST,Totals_STL,Totals_BLK,Totals_TOV,Totals_PF,Possessions,Off_Eff,Poss_Per_Game
0,1.0,Abilene Christian,33.0,14,19,.424,-7.38,-1.45,NaN,5,...,369.0,1047,440,321,91,472.0,691,2294.975,1.011340,69.544697
1,2.0,Air Force,32.0,3,29,.094,-13.75,4.60,NaN,0,...,225.0,931,389,180,76,472.0,570,2156.650,0.912990,67.395313
2,3.0,Akron NCAA,35.0,29,6,.829,8.48,-2.91,NaN,17,...,397.0,1319,639,261,109,381.0,549,2505.975,1.227067,71.599286
3,4.0,Alabama NCAA,35.0,25,10,.714,23.03,14.57,NaN,13,...,437.0,1425,571,225,174,343.0,646,2613.200,1.221491,74.662857
4,5.0,Alabama A&M,33.0,18,15,.545,-13.31,-10.81,NaN,10,...,311.0,1126,387,198,93,376.0,607,2246.800,1.058394,68.084848


In [6]:
df_combine = pd.DataFrame({
    'Team': df_sb['Unnamed: 1_level_0_School'],
    'Offensive': round(df_sb['Off_Eff'] * 100, 2),
    'Defensive': round(df_ob['Def_Eff'] * 100, 2),
    'Possessions': df_sb['Poss_Per_Game']
})

df_combine

,Team,Offensive,Defensive,Possessions
0,Abilene Christian,101.13,104.05,69.544697
1,Air Force,91.30,118.51,67.395313
2,Akron NCAA,122.71,103.33,71.599286
3,Alabama NCAA,122.15,110.65,74.662857
4,Alabama A&M,105.84,106.73,68.084848
...,...,...,...,...
378,Wright State NCAA,116.50,106.47,69.111429
379,Wyoming,113.20,107.94,67.914394
380,Xavier,109.42,112.18,71.616667
381,Yale,121.75,105.91,66.770161


In [7]:
df_combine['Team'] = df_combine['Team'].str.replace(r'\s*NCAA$', '', regex=True)

# Also clear out any asterisks if Sports-Reference added them
df_combine['Team'] = df_combine['Team'].str.replace(r'\*$', '', regex=True)

# Finally, strip any accidental trailing/leading spaces left over
df_combine['Team'] = df_combine['Team'].str.strip()

# Check your clean school names
print(df_combine['Team'].head(10))

0    Abilene Christian
1            Air Force
2                Akron
3              Alabama
4          Alabama A&M
5        Alabama State
6          Albany (NY)
7         Alcorn State
8             American
9    Appalachian State
Name: Team, dtype: object


In [8]:
df_combine

,Team,Offensive,Defensive,Possessions
0,Abilene Christian,101.13,104.05,69.544697
1,Air Force,91.30,118.51,67.395313
2,Akron,122.71,103.33,71.599286
3,Alabama,122.15,110.65,74.662857
4,Alabama A&M,105.84,106.73,68.084848
...,...,...,...,...
378,Wright State,116.50,106.47,69.111429
379,Wyoming,113.20,107.94,67.914394
380,Xavier,109.42,112.18,71.616667
381,Yale,121.75,105.91,66.770161


In [9]:
df_sb["Overall_SOS"] = pd.to_numeric(df_sb["Overall_SOS"], errors='coerce')

In [12]:
df_combine["Offensive"] = round(df_sb["Off_Eff"] * ((df_sb["Overall_SOS"]/200) + 1) * 100,2)
df_combine["Defensive"] = round(df_ob["Def_Eff"] * ((df_sb["Overall_SOS"]/200) - 1) * -100,2)
df_combine["Net"] = round(df_combine["Offensive"] - df_combine["Defensive"],2)
df_combine["Possessions"] = round(df_combine["Possessions"], 2)
df_combine.insert(0, "Net", df_combine.pop("Net"))
df_combine.insert(0, "Team", df_combine.pop("Team"))
df_combine.head()

,Team,Net,Offensive,Defensive,Possessions
0,Abilene Christian,-4.41,100.40,104.81,69.54
1,Air Force,-22.39,93.40,115.79,67.40
2,Akron,16.09,120.92,104.83,71.60
3,Alabama,28.46,131.05,102.59,74.66
4,Alabama A&M,-12.38,100.12,112.50,68.08


In [14]:
df_combine.to_csv('output2.csv', index=False)